# **Converting a CNN Architecture Diagram into PyTorch**

### **Laboratory Exercise 5** - *Deep Learning Fundamentals*
*From the course module:* ***E1 - CNN Implementation*** *(Laboratory Task 6)*

**Instruction:** Convert the following CNN architecture diagram into a PyTorch CNN Architecture.

```mermaid
graph TD
    Input["Input (1, 28, 28)"] --> B1

    subgraph Convolutional Layers
        B1["ReLU(Conv1 + MaxPool1)<br>Conv1: kernel=(3,3), stride=1, pad=1<br>MaxPool1: kernel=(2,2), stride=2, pad=1<br>shape = (32, 32, 28, 28)"] --> B2
        B2["ReLU(Conv2)<br>kernel=(3,3), stride=1, pad=1<br>shape = (32, 64, ?, ?)"] --> B3
        B3["ReLU(Conv3)<br>kernel=(3,3), stride=1, pad=1<br>shape = (64, 128, ?, ?)"] --> B4
        B4["ReLU(Conv4 + MaxPool2)<br>Conv4: kernel=(3,3), stride=1, pad=1<br>MaxPool2: kernel=(2,2), stride=2, pad=0<br>shape = (128, 256, ?, ?)"]
    end

    B4 --> Flatten

    subgraph Dense / Fully Connected Layers
        Flatten["DropOut (p=0.2)<br>Flatten Input<br>shape = (32, ?)"] --> FCN1
        FCN1["ReLU(FCN1)<br>input=?, output=1000"] --> FCN2
        FCN2["ReLU(FCN2)<br>input=1000, output=500"] --> FCN3
        FCN3["SoftMax(FCN3)<br>input=500, output=?"]
    end
```

The diagram leaves several values as `?`. They must be **calculated** from the layer settings before the model can be written. For both convolution and pooling layers, the output size along one spatial dimension is:

$$
\text{out}=\left\lfloor\frac{\text{in}+2p-k}{s}\right\rfloor+1
$$

where $k$ is the kernel size, $s$ the stride, and $p$ the padding.

## **Implementation**



### **Import Libraries**

In [1]:
!pip install torch

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### **Define the CNN Architecture**

Each box in the diagram becomes a layer, named after the box. `forward()` follows the diagram from top to bottom: `ReLU( ... )` wraps each Conv and the first two FC layers, and Softmax is applied to the last FC layer.

In [3]:
class CNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()

        # Convolution block 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)

        # Convolution block 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        # Regularization and flattening
        self.dropout = nn.Dropout(p=0.2)
        self.flatten = nn.Flatten()

        # Fully connected layers
        self.fcn1 = nn.Linear(256 * 7 * 7, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.dropout(x)
        x = self.flatten(x)

        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)
        return x

model = CNN(num_classes=10)
print(model)

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fcn1): Linear(in_features=12544, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)


### **Verify the Shapes**

A dummy batch of 32 images (1×28×28) is passed through the model. A forward hook on each layer prints its output shape, and the values are compared against the table computed by hand.

In [4]:
expected = {
    "conv1":   (32, 32, 28, 28),
    "pool1":   (32, 32, 15, 15),
    "conv2":   (32, 64, 15, 15),
    "conv3":   (32, 128, 15, 15),
    "conv4":   (32, 256, 15, 15),
    "pool2":   (32, 256, 7, 7),
    "dropout": (32, 256, 7, 7),
    "flatten": (32, 12544),
    "fcn1":    (32, 1000),
    "fcn2":    (32, 500),
    "fcn3":    (32, 10),
}

def make_hook(name):
    def hook(module, inputs, output):
        shape = tuple(output.shape)
        status = "OK" if shape == expected[name] else "MISMATCH"
        print(f"{name:8s} -> {str(shape):22s} {status}")
        assert shape == expected[name]
    return hook

for name in expected:
    getattr(model, name).register_forward_hook(make_hook(name))

dummy = torch.randn(32, 1, 28, 28)
out = model(dummy)

print("\nOutput shape:", tuple(out.shape))
print("Each row sums to 1 (Softmax):", torch.allclose(out.sum(dim=1), torch.ones(32)))

conv1    -> (32, 32, 28, 28)       OK
pool1    -> (32, 32, 15, 15)       OK
conv2    -> (32, 64, 15, 15)       OK
conv3    -> (32, 128, 15, 15)      OK
conv4    -> (32, 256, 15, 15)      OK
pool2    -> (32, 256, 7, 7)        OK
dropout  -> (32, 256, 7, 7)        OK
flatten  -> (32, 12544)            OK
fcn1     -> (32, 1000)             OK
fcn2     -> (32, 500)              OK
fcn3     -> (32, 10)               OK

Output shape: (32, 10)
Each row sums to 1 (Softmax): True


### **Count the Parameters**

In [5]:
total = 0
for name, p in model.named_parameters():
    total += p.numel()
    print(f"{name:14s} {str(tuple(p.shape)):20s} {p.numel():>12,}")

print(f"\nTotal trainable parameters: {total:,}")

conv1.weight   (32, 1, 3, 3)                 288
conv1.bias     (32,)                          32
conv2.weight   (64, 32, 3, 3)             18,432
conv2.bias     (64,)                          64
conv3.weight   (128, 64, 3, 3)            73,728
conv3.bias     (128,)                        128
conv4.weight   (256, 128, 3, 3)          294,912
conv4.bias     (256,)                        256
fcn1.weight    (1000, 12544)          12,544,000
fcn1.bias      (1000,)                     1,000
fcn2.weight    (500, 1000)               500,000
fcn2.bias      (500,)                        500
fcn3.weight    (10, 500)                   5,000
fcn3.bias      (10,)                          10

Total trainable parameters: 13,438,350


## **Final Answer**

Values that were `?` in the diagram:

| Location | Value |
|---|---|
| Conv2, Conv3, Conv4 output size | (…, 15, 15) |
| MaxPool2 output size | (…, 7, 7) |
| Flatten shape | (32, 12544) |
| FCN1 input | 12544 |
| FCN3 output | 10 |

**Interpretation**

The first pooling layer uses padding=1, so it turns 28×28 into 15×15 instead of 14×14. The three convolutions with k=3, s=1, p=1 keep that size, while the number of channels grows from 32 to 256. The second pooling layer (padding 0) reduces 15×15 to 7×7, so the flattened vector has $256\times7\times7=12544$ values, which is the input size of FCN1. Almost all the parameters sit in FCN1, since it connects 12544 inputs to 1000 neurons.

**Conclusion:**

The diagram was converted into a PyTorch `nn.Module` with 4 convolution layers, 2 max-pooling layers, dropout, flatten, and 3 fully connected layers. The forward-hook check confirms that every layer produces the shape expected from the diagram.

**Note on training:** the model ends with Softmax because the diagram shows it.